# 💻 Unidad 1: Material Complementario - Práctica
## Módulo 01 - EDA Avanzado con Profiling
### Laboratorio (Herramientas) - Universidad del Aconcagua

---

## 🎯 Objetivos de la Práctica

En esta práctica vas a:

1. ✅ Instalar y configurar ydata-profiling y sweetviz
2. ✅ Generar reportes automáticos de EDA con los datos de la panadería
3. ✅ Comparar datasets (train vs test) con Sweetviz
4. ✅ Analizar relaciones con variable target
5. ✅ Interpretar warnings y métricas automáticas
6. ✅ Tomar decisiones de limpieza basadas en profiling

---

### 📋 Ejercicios

1. **Ejercicio 1**: Profiling completo del dataset de ventas con ydata-profiling
2. **Ejercicio 2**: Comparación Train/Test con Sweetviz
3. **Ejercicio 3**: Target Analysis con Sweetviz
4. **Ejercicio 4**: Interpretación de warnings y acciones correctivas
5. **Ejercicio 5**: Profiling de datos limpios vs crudos

---

### ⏱️ Duración Estimada: 60 minutos

## 🛠️ Setup: Instalación de Librerías

Primero instalamos las herramientas de profiling:

In [0]:
# Instalar librerías de profiling
# NOTA: ydata-profiling no está disponible en ARM64 (serverless) debido a dependencias de compilación
# Como alternativa, usamos sweetviz y dtale que funcionan sin problemas
%pip install sweetviz dtale

print("✅ Librerías instaladas: sweetviz y dtale")
print("⚠️ Nota: ydata-profiling no está disponible en este entorno ARM64")
print("   Usaremos sweetviz y dtale como alternativas igualmente poderosas")

In [0]:
# Imports
import pandas as pd
import numpy as np
# ydata_profiling no está disponible en ARM64 serverless
# Usamos sweetviz y dtale como alternativas
import sweetviz as sv
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

---

## 📊 Ejercicio 1: Profiling Completo con ydata-profiling

**Objetivo**: Generar un reporte exhaustivo del dataset de ventas

**Dataset**: Ventas de la panadería (50,000+ transacciones)

**Tarea**:
1. Cargar el dataset de ventas
2. Generar reporte con ydata-profiling
3. Analizar las secciones del reporte
4. Identificar problemas de calidad de datos

In [0]:
# Cargar dataset de ventas
ventas = pd.read_csv("/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/ventas.csv")

print(f"📊 Dataset cargado:")
print(f"  • Filas: {ventas.shape[0]:,}")
print(f"  • Columnas: {ventas.shape[1]}")
print(f"\n🔍 Primeras filas:")
display(ventas.head())

In [0]:
# Generar reporte EDA manual (ydata-profiling y sweetviz no están disponibles en ARM64)
print("📊 REPORTE DE PROFILING - Dataset Ventas\n" + "="*60)

# Overview
print(f"\n📁 Overview:")
print(f"  • Variables: {ventas.shape[1]}")
print(f"  • Observaciones: {ventas.shape[0]:,}")
print(f"  • Duplicados: {ventas.duplicated().sum()} ({ventas.duplicated().mean():.1%})")
print(f"  • Missing cells: {ventas.isnull().sum().sum()} ({ventas.isnull().sum().sum()/(ventas.shape[0]*ventas.shape[1]):.1%})")

# Tipos de variables
print(f"\n📊 Tipos de variables:")
for dtype, count in ventas.dtypes.value_counts().items():
    print(f"  • {dtype}: {count}")

# Estadísticas por columna
print(f"\n📈 Estadísticas descriptivas:")
display(ventas.describe())

print(f"\n✅ Análisis completado")

In [0]:
# Análisis detallado de calidad de datos
print("📊 ANÁLISIS DETALLADO DE CALIDAD\n" + "="*60)

# Missing values por columna
print(f"\n❌ Missing Values por Columna:")
missing = ventas.isnull().sum()
missing_pct = (missing / len(ventas) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing': missing,
    'Porcentaje': missing_pct
}).sort_values('Missing', ascending=False)

for col, row in missing_df[missing_df['Missing'] > 0].iterrows():
    print(f"  • {col}: {int(row['Missing'])} ({row['Porcentaje']}%)")

if missing_df['Missing'].sum() == 0:
    print("  ✅ Sin valores faltantes")

# Cardinalidad (valores únicos)
print(f"\n🔢 Cardinalidad de Variables:")
for col in ventas.columns[:8]:  # Primeras 8 columnas
    unique_count = ventas[col].nunique()
    unique_pct = round((unique_count / len(ventas) * 100), 1)
    print(f"  • {col}: {unique_count} valores únicos ({unique_pct}%)")

print(f"\n✅ Análisis completado")

---

## 🔄 Ejercicio 2: Comparación Train/Test con Sweetviz

**Objetivo**: Validar que el split train/test tiene distribuciones similares

**Tarea**:
1. Hacer split 80/20 de los datos
2. Generar reporte comparativo con Sweetviz
3. Analizar diferencias entre train y test
4. Validar que no hay data leakage

In [0]:
# Hacer split train/test (80/20)
train, test = train_test_split(ventas, test_size=0.2, random_state=42)

print("✂️ Split Train/Test completado:")
print(f"  • Train: {len(train):,} filas ({len(train)/len(ventas):.0%})")
print(f"  • Test: {len(test):,} filas ({len(test)/len(ventas):.0%})")
print(f"  • Total: {len(ventas):,} filas")

In [0]:
# Comparación manual Train vs Test (Sweetviz no compatible con ARM64)
print("📊 COMPARACIÓN TRAIN VS TEST\n" + "="*60)

# Comparar distribuciones de columnas numéricas
print("\n📈 Estadísticas Comparativas (Columnas Numéricas):\n")
numeric_cols = train.select_dtypes(include=[np.number]).columns[:5]

for col in numeric_cols:
    print(f"\n{col}:")
    print(f"  Train - Media: {train[col].mean():.2f} | Mediana: {train[col].median():.2f} | Std: {train[col].std():.2f}")
    print(f"  Test  - Media: {test[col].mean():.2f} | Mediana: {test[col].median():.2f} | Std: {test[col].std():.2f}")
    
    # Diferencia porcentual en medias
    diff_pct = abs((train[col].mean() - test[col].mean()) / train[col].mean() * 100)
    status = "✅" if diff_pct < 5 else "⚠️"
    print(f"  {status} Diferencia en media: {diff_pct:.1f}%")

print(f"\n✅ Comparación completada")
print("\n💡 Interpretación: Diferencias < 5% indican distribuciones similares")

In [0]:
# Comparación estadística de distribuciones
import scipy.stats as stats

print("📊 COMPARACIÓN ESTADÍSTICA TRAIN VS TEST\n" + "="*60)

# Columnas numéricas a comparar
numeric_cols = ventas.select_dtypes(include=[np.number]).columns

for col in numeric_cols[:5]:  # Primeras 5 columnas numéricas
    # Test de Kolmogorov-Smirnov (similitud de distribuciones)
    statistic, pvalue = stats.ks_2samp(train[col], test[col])
    
    similar = "✅" if pvalue > 0.05 else "⚠️"
    print(f"\n{similar} {col}:")
    print(f"  • Media Train: {train[col].mean():.2f}")
    print(f"  • Media Test: {test[col].mean():.2f}")
    print(f"  • KS p-value: {pvalue:.4f} {'(Similar)' if pvalue > 0.05 else '(Diferente)'}")

print("\n" + "="*60)
print("💡 Interpretación:")
print("  • p-value > 0.05: Distribuciones similares ✅")
print("  • p-value < 0.05: Distribuciones diferentes ⚠️ (posible problema)")

---

## 🎯 Ejercicio 3: Target Analysis con Sweetviz

**Objetivo**: Analizar qué variables están más correlacionadas con una variable objetivo

**Escenario**: Queremos predecir si una venta es "alta" o "baja"

**Tarea**:
1. Crear variable target binaria (`venta_alta`)
2. Analizar con Sweetviz usando target_feat
3. Identificar features más importantes

In [0]:
# Crear variable target: Venta alta si total > mediana
median_total = ventas['total'].median()
ventas_target = ventas.copy()
ventas_target['venta_alta'] = (ventas_target['total'] > median_total).astype(int)

print(f"🎯 Variable target creada: 'venta_alta'")
print(f"  • Umbral: ${median_total:.2f}")
print(f"  • Distribución:")
print(ventas_target['venta_alta'].value_counts())
print(f"\n  • Clase 0 (Baja): {(ventas_target['venta_alta']==0).sum()} ventas ({(ventas_target['venta_alta']==0).mean():.1%})")
print(f"  • Clase 1 (Alta): {(ventas_target['venta_alta']==1).sum()} ventas ({(ventas_target['venta_alta']==1).mean():.1%})")

In [0]:
# Análisis de Target Manual (Sweetviz no compatible con ARM64)
print("🎯 TARGET ANALYSIS - Variable 'venta_alta'\n" + "="*60)

# Análisis de distribución por target
print("\n📊 Distribución del Target:")
print(ventas_target['venta_alta'].value_counts())
print(f"\nBalance: {ventas_target['venta_alta'].value_counts(normalize=True).round(3).to_dict()}")

# Comparar estadísticas por clase
print("\n📈 Comparación de Estadísticas por Clase (Target):")
for col in ['total', 'sucursal_id', 'mes']:
    print(f"\n{col}:")
    print(f"  Clase 0 - Media: {ventas_target[ventas_target['venta_alta']==0][col].mean():.2f}")
    print(f"  Clase 1 - Media: {ventas_target[ventas_target['venta_alta']==1][col].mean():.2f}")
    
print(f"\n✅ Target Analysis completado")
print("\n💡 Interpretación: Diferencias grandes entre clases indican features predictivos")

In [0]:
# Calcular correlaciones con el target
print("📊 CORRELACIONES CON TARGET 'venta_alta'\n" + "="*60)

# Seleccionar solo columnas numéricas
numeric_df = ventas_target.select_dtypes(include=[np.number])

# Calcular correlaciones
correlations = numeric_df.corr()['venta_alta'].sort_values(ascending=False)

print("\n🔼 Top 10 variables MÁS correlacionadas con venta_alta:\n")
for i, (col, corr) in enumerate(correlations.head(11).items(), 1):
    if col != 'venta_alta':  # Excluir la variable misma
        icon = "🟢" if corr > 0.3 else "🟡" if corr > 0.1 else "⚪"
        print(f"{i}. {icon} {col:20s} = {corr:+.3f}")

print("\n\n💡 Interpretación:")
print("  • 🟢 |corr| > 0.3: Correlación fuerte (importante para el modelo)")
print("  • 🟡 |corr| > 0.1: Correlación moderada (potencialmente útil)")
print("  • ⚪ |corr| < 0.1: Correlación débil (poco predictivo)")

---

## ⚠️ Ejercicio 4: Interpretación de Warnings y Acciones Correctivas

**Objetivo**: Aprender a interpretar warnings automáticos y tomar acciones

**Tarea**:
1. Simular problemas comunes de datos
2. Generar profiling y analizar warnings
3. Aplicar correcciones basadas en warnings

In [0]:
# Crear dataset con problemas simulados
ventas_problemas = ventas.copy().head(1000)

# Problema 1: Columna constante (sin variación)
ventas_problemas['columna_constante'] = 'VALOR_FIJO'

# Problema 2: Alta cardinalidad (muchos valores únicos)
ventas_problemas['id_unico'] = range(len(ventas_problemas))

# Problema 3: Alta correlación (multicolinealidad)
ventas_problemas['total_duplicado'] = ventas_problemas['total'] * 1.01  # Casi idéntico

# Problema 4: Muchos zeros
ventas_problemas['columna_ceros'] = 0
ventas_problemas.loc[ventas_problemas.index[:50], 'columna_ceros'] = [1,2,3]*16 + [1,2]

print("⚠️ Dataset con problemas simulados creado:")
print(f"  • Filas: {len(ventas_problemas)}")
print(f"  • Columnas: {ventas_problemas.shape[1]}")
print(f"  • Problemas introducidos: 4 tipos")

In [0]:
# Análisis manual de problemas de datos (ydata-profiling no disponible en ARM64)
print("🔄 Analizando problemas de datos...\n" + "="*60)

# Detectar problemas automáticamente
warnings_count = 0
warnings = []

# 1. Detectar columnas constantes (sin variación)
for col in ventas_problemas.columns:
    if ventas_problemas[col].nunique() == 1:
        warnings.append(f"Columna constante: '{col}' → No aporta información")
        warnings_count += 1

# 2. Detectar alta cardinalidad (muchos valores únicos)
for col in ventas_problemas.columns:
    unique_ratio = ventas_problemas[col].nunique() / len(ventas_problemas)
    if unique_ratio > 0.95:  # Más del 95% de valores únicos
        warnings.append(f"Alta cardinalidad: '{col}' ({unique_ratio:.1%} valores únicos) → Posible ID")
        warnings_count += 1

# 3. Detectar alta correlación (multicolinealidad)
numeric_cols = ventas_problemas.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 1:
    corr_matrix = ventas_problemas[numeric_cols].corr()
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.95:  # Correlación muy alta
                warnings.append(f"Alta correlación: '{corr_matrix.columns[i]}' <-> '{corr_matrix.columns[j]}' (r={corr_matrix.iloc[i,j]:.3f}) → Redundancia")
                warnings_count += 1

# 4. Detectar columnas con muchos ceros
for col in numeric_cols:
    zero_ratio = (ventas_problemas[col] == 0).mean()
    if zero_ratio > 0.9:  # Más del 90% ceros
        warnings.append(f"Muchos ceros: '{col}' ({zero_ratio:.1%} ceros) → Datos sparse")
        warnings_count += 1

print(f"\n⚠️ WARNINGS DETECTADOS: {warnings_count}\n" + "="*60)

if warnings_count > 0:
    print("\nEl análisis detectó los siguientes problemas:\n")
    for i, warning in enumerate(warnings, 1):
        print(f"{i}. {warning}")
    
    print("\n\n💡 Problemas típicos detectados:")
    print("  1. Columnas constantes (sin variación) → No aportan información")
    print("  2. Alta cardinalidad → Posibles IDs, no features")
    print("  3. Alta correlación → Multicolinealidad, redundancia")
    print("  4. Muchos ceros/nulos → Datos sparse, imputar o eliminar")
    print("  5. Outliers extremos → Revisar y decidir tratamiento")
else:
    print("\n✅ No se detectaron problemas automáticos")

print(f"\n✅ Análisis completado")

In [0]:
# Aplicar correcciones basadas en warnings
print("🛠️ APLICANDO CORRECCIONES\n" + "="*60)

ventas_corregido = ventas_problemas.copy()

# Corrección 1: Eliminar columna constante
ventas_corregido = ventas_corregido.drop('columna_constante', axis=1)
print("✅ 1. Columna constante eliminada")

# Corrección 2: Eliminar columna con alta cardinalidad (ID)
ventas_corregido = ventas_corregido.drop('id_unico', axis=1)
print("✅ 2. Columna de alta cardinalidad (ID) eliminada")

# Corrección 3: Eliminar columna altamente correlacionada
ventas_corregido = ventas_corregido.drop('total_duplicado', axis=1)
print("✅ 3. Columna redundante (alta correlación) eliminada")

# Corrección 4: Evaluar columna con muchos ceros
zero_pct = (ventas_corregido['columna_ceros'] == 0).mean()
if zero_pct > 0.9:  # Si >90% son ceros
    ventas_corregido = ventas_corregido.drop('columna_ceros', axis=1)
    print(f"✅ 4. Columna con {zero_pct:.1%} ceros eliminada")

print(f"\n📊 Resultado:")
print(f"  • Columnas antes: {ventas_problemas.shape[1]}")
print(f"  • Columnas después: {ventas_corregido.shape[1]}")
print(f"  • Columnas eliminadas: {ventas_problemas.shape[1] - ventas_corregido.shape[1]}")

---

## 🔄 Ejercicio 5: Comparación Antes/Después de Limpieza

**Objetivo**: Validar mejoras después de limpiar datos

**Tarea**:
1. Comparar dataset original vs corregido con Sweetviz
2. Verificar reducción de warnings
3. Documentar mejoras

In [0]:
# Comparación manual Antes vs Después (Sweetviz incompatible con NumPy 2.0+)
print("🔄 COMPARACIÓN ANTES VS DESPUÉS DE LIMPIEZA\n" + "="*60)

# Comparar dimensiones
print("\n📊 Dimensiones:")
print(f"  Antes:   {ventas_problemas.shape[0]} filas × {ventas_problemas.shape[1]} columnas")
print(f"  Después: {ventas_corregido.shape[0]} filas × {ventas_corregido.shape[1]} columnas")
print(f"  Cambio:  Eliminadas {ventas_problemas.shape[1] - ventas_corregido.shape[1]} columnas")

# Comparar missing values
print("\n❌ Missing Values:")
missing_antes = ventas_problemas.isnull().sum().sum()
missing_despues = ventas_corregido.isnull().sum().sum()
print(f"  Antes:   {missing_antes} celdas vacías")
print(f"  Después: {missing_despues} celdas vacías")

# Comparar estadísticas de columnas numéricas comunes
common_numeric = list(set(ventas_problemas.select_dtypes(include=[np.number]).columns) & 
                      set(ventas_corregido.select_dtypes(include=[np.number]).columns))

if common_numeric:
    print("\n📈 Estadísticas de Columnas Comunes (primeras 3):")
    for col in common_numeric[:3]:
        print(f"\n  {col}:")
        print(f"    Antes  - Media: {ventas_problemas[col].mean():.2f}, Std: {ventas_problemas[col].std():.2f}")
        print(f"    Después - Media: {ventas_corregido[col].mean():.2f}, Std: {ventas_corregido[col].std():.2f}")

print("\n✅ Comparación completada")
print("\n💡 Interpretación: El dataset corregido eliminó columnas problemáticas")
print("   manteniendo la calidad de las columnas relevantes")

In [0]:
# Calcular métricas de mejora
print("📊 MEJORAS CUANTIFICADAS\n" + "="*60)

print("\n📉 Reducción de dimensionalidad:")
print(f"  • Columnas antes: {ventas_problemas.shape[1]}")
print(f"  • Columnas después: {ventas_corregido.shape[1]}")
print(f"  • Columnas eliminadas: {ventas_problemas.shape[1] - ventas_corregido.shape[1]}")
print(f"  • Reducción: {(1 - ventas_corregido.shape[1]/ventas_problemas.shape[1]):.1%}")

print("\n✅ Beneficios de la limpieza:")
print("  1. Menos features irrelevantes → Modelos más simples")
print("  2. Sin multicolinealidad → Mejor interpretabilidad")
print("  3. Sin columnas constantes → No desperdicia recursos")
print("  4. Dataset más limpio → Mejor performance de ML")

print("\n💡 Próximos pasos recomendados:")
print("  • Guardar dataset corregido como nueva versión")
print("  • Documentar transformaciones aplicadas")
print("  • Validar con stakeholders")
print("  • Proceder con feature engineering")

---

## ✅ Resumen de la Práctica

### 🎯 Lo que Aprendiste

#### **ydata-profiling**
* ✅ Generar reportes exhaustivos de EDA automáticamente
* ✅ Extraer métricas clave programáticamente
* ✅ Identificar warnings de calidad de datos
* ✅ Documentar datasets para el equipo

#### **Sweetviz**
* ✅ Comparar train/test para validar splits
* ✅ Analizar asociaciones con variable target
* ✅ Comparar datos antes/después de limpieza
* ✅ Generar reportes visuales rápidamente

#### **Workflow de EDA Avanzado**
1. **Profiling inicial** con ydata-profiling → Entender el dataset completo
2. **Train/Test split** y validación con Sweetviz → Asegurar distribuciones similares
3. **Target analysis** con Sweetviz → Identificar features importantes
4. **Interpretación de warnings** → Detectar problemas automáticamente
5. **Limpieza y validación** → Comparar antes/después

---

### 📊 Comparación: EDA Manual vs Automatizado

| Aspecto | EDA Manual | EDA Automatizado |
|---------|-----------|------------------|
| **Tiempo** | 30-60 min | 2-5 min |
| **Profundidad** | Depende del analista | Siempre completo |
| **Reproducibilidad** | Baja | Alta |
| **Warnings** | Manual, puede fallar | Automático |
| **Comparaciones** | Complejo | 1 línea de código |
| **Documentación** | Manual | HTML compartible |

---

### 💡 Mejores Prácticas Aprendidas

✅ **Siempre profiling primero** antes de transformar datos  
✅ **Guardar reportes HTML** con timestamp para trazabilidad  
✅ **Revisar warnings** automáticamente, no ignorarlos  
✅ **Comparar train/test** para detectar data leakage  
✅ **Target analysis** para feature selection temprana  
✅ **Validar limpieza** comparando antes/después  
✅ **Integrar en pipelines** para monitoreo continuo

---

### 🚀 Próximos Pasos

**En el curso:**
* ✅ Aplica profiling en tus TPs (TP02, TP03)
* ✅ Continúa con Módulo 02: Manejo de Datos Faltantes y Outliers
* ✅ Usa profiling en el Proyecto Final

**Fuera del curso:**
* 📖 Revisa documentación de [ydata-profiling](https://docs.profiling.ydata.ai/)
* 📖 Explora [Sweetviz](https://github.com/fbdesignpro/sweetviz)
* 💻 Practica con tus propios datasets

---

### 🎓 ¡Felicitaciones!

Completaste exitosamente el Módulo 01 de Material Complementario.  
Ahora dominas herramientas profesionales de EDA automatizado que te ahorrarán horas de trabajo en proyectos reales.

---

**Universidad del Aconcagua - Facultad de Ciencias Económicas y Jurídicas**  
**Licenciatura en Analítica de Negocios**  
**Mendoza, Argentina 🇦🇷**